In [1]:
from torch import nn
import numpy as np
import matplotlib as mlp
import pandas as pd
import h5py
from time import sleep
from torch.utils.data import Dataset, DataLoader, random_split, Subset

In [2]:
from model import H5Data
path = "./data.hdf5"
data = H5Data(path)

In [3]:
from sklearn.model_selection import train_test_split

labels = [int(float(group)) for group, _ in data.index_map]
seed = 1
train_idx, test_idx = train_test_split(
    range(len(data.index_map)),
    train_size=0.8,
    stratify=labels,
    random_state=seed
)
train_sub = Subset(data, train_idx)
test_sub = Subset(data, test_idx)

BATCH_SIZE = 32
train_dataloader = DataLoader(train_sub, batch_size=BATCH_SIZE, shuffle=True)
test_dataloader = DataLoader(test_sub, batch_size=BATCH_SIZE)
print(f"Length of train dataloader: {len(train_dataloader)}")
print(f"Length of test dataloader: {len(test_dataloader)}")

Length of train dataloader: 81261
Length of test dataloader: 20316


In [4]:
import torch
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

In [7]:
import torch.optim as optim
import optuna
from model import star_tracker_v1

def objective(trial):
    hidden_size = trial.suggest_int('hidden_size', 16, 256)
    n_bins = 25
    learning_rate = trial.suggest_float('lr', 1e-4, 1e-1, log=True)
    
    #(bins classes hidden)
    model = star_tracker_v1(n_bins, 9029, hidden_size)
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_fn = nn.CrossEntropyLoss()

    epochs = 10
    for epoch in range(epochs):
        model.train()
        for batch, (x, y) in enumerate(train_dataloader):
            x, y = x.to(device), y.to(device)
    
            y_pred = model(x)
            loss = loss_fn(y_pred, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
    test_loss = 0
    model.eval()
    with torch.inference_mode():
        for x,y in (test_dataloader):
            x, y = x.to(device), y.to(device)

            y_pred = model(x)
            test_loss += loss_fn(y_pred, y).item()

        test_loss /= len(test_dataloader)

    trial.report(test_loss, epoch)
    if trial.should_prune():
        raise optuna.exceptions.TrialPruned()
    return test_loss

In [8]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)

[I 2026-04-12 23:25:59,026] A new study created in memory with name: no-name-a4de23b0-94fc-4338-a313-67352b4599f0
[W 2026-04-12 23:46:30,492] Trial 0 failed with parameters: {'hidden_size': 134, 'lr': 0.0001361595594039674} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/Users/kadenwu/miniconda3/envs/misc/lib/python3.12/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/var/folders/vr/lxk8rp6n0871gdlxdyd9kw3m0000gn/T/ipykernel_19413/2710625615.py", line 22, in objective
    y_pred = model(x)
             ^^^^^^^^
  File "/Users/kadenwu/miniconda3/envs/misc/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1532, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/kadenwu/miniconda3/envs/misc/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1541, in _

KeyboardInterrupt: 

In [ ]:
import torch
from model import star_tracker_v1

device = torch.device("mps" if torch.backends.mps.is_a_available() else "cpu")

model = star_tracker_v1(25, 9029, 60)
model.to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params=model.parameters(), lr=0.01)

In [ ]:
epochs = 100
for epoch in range(epochs):
    print(f"Epoch: {epoch}")

    train_loss, train_accuracy = 0, 0
    model.train()
    for batch, (x, y) in enumerate(train_dataloader):
        x, y = x.to(device), y.to(device)

        y_pred = model(x)
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()
        train_pred = torch.argmax(y_pred, dim=1)
        train_accuracy += (train_pred == y).sum().item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss /= len(train_dataloader)
    train_accuracy /= len(train_dataloader) * BATCH_SIZE

    test_loss, test_accuracy = 0, 0
    model.eval()
    with torch.inference_mode():
        for x,y in (test_dataloader):
            x, y = x.to(device), y.to(device)

            y_pred = model(x)
            test_loss += loss_fn(y_pred, y).item()
            test_pred = torch.argmax(y_pred, dim=1)
            test_accuracy += (test_pred == y).sum().item()

        test_loss /= len(test_dataloader)
        test_accuracy /= len(test_dataloader) * BATCH_SIZE

    print(f"\nTrain loss: {train_loss:.5f} | Train Accuracy: {train_accuracy:.5f} | Test loss: {test_loss:.5f}, Test acc: {test_accuracy:.2f}\n")

Epoch: 0
